# 智慧垃圾分類 — YOLO26m 訓練 Notebook (v6, Kaggle)

**ADR-001 對齊：YOLO26m + TensorRT FP16 部署**

**執行前必做：Settings → Accelerator → GPU T4 x2（或 P100）**

## 與 Colab 版差異
- 輸出自動保存至 `/kaggle/working/`，不需要 Google Drive
- Roboflow API Key 請加入 Kaggle Secrets（Add-ons → Secrets → New Secret，名稱：`ROBOFLOW_API_KEY`）
- epochs=80, patience=20（比 Colab 版更充分）
- 加入 copy_paste 增強改善少數類別（塑膠袋、寶特瓶）

## 五大類別
| ID | 中文 | GPIO Pin | LED |
|----|------|----------|-----|
| 0 | 寶特瓶 | Pin 11 | 綠 |
| 1 | 鐵鋁罐 | Pin 13 | 黃 |
| 2 | 紙餐盒 | Pin 15 | 藍 |
| 3 | 塑膠袋 | Pin 21 | 白 |
| 4 | 一般垃圾 | Pin 23 | 紅 |

In [ ]:
# ── Cell 1：確認 GPU ───────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    raise RuntimeError('❌ 沒有 GPU，請到 Settings → Accelerator → GPU T4 x2')

In [ ]:
# ── Cell 2：安裝套件 ───────────────────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', 'ultralytics', 'roboflow', 'pyyaml', '-q'], check=True)
print('套件安裝完成')

In [ ]:
# ── Cell 3：Clone 專案 ─────────────────────────────────────────────────────────
import os

WORKDIR = '/kaggle/working/AI-course'
if os.path.exists(WORKDIR):
    print('Repo 已存在，執行 git pull...')
    os.system(f'cd {WORKDIR} && git pull')
else:
    os.system('cd /kaggle/working && git clone https://github.com/Saibusu/AI-course.git')

os.chdir(WORKDIR)
print('工作目錄：', os.getcwd())

In [ ]:
# ── Cell 4：下載 Roboflow 資料集 ───────────────────────────────────────────────
# API Key 從 Kaggle Secrets 讀取
# 設定方式：右上角 Add-ons → Secrets → New Secret
#           Name: ROBOFLOW_API_KEY  Value: (你的 API Key)
import os
from roboflow import Roboflow

try:
    from kaggle_secrets import UserSecretsClient
    ROBOFLOW_API_KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
    print('✅ API Key 從 Kaggle Secrets 讀取成功')
except Exception:
    ROBOFLOW_API_KEY = 'YOUR_ROBOFLOW_API_KEY'  # ← 備用：直接填入，勿上傳 GitHub
    print('⚠️  請確認 Kaggle Secrets 設定，或直接填入 API Key')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('projectverba').project('yolo-waste-detection')
version = project.version(1)
dataset = version.download('yolo26', location='data/roboflow_raw')

print('下載完成，路徑：', dataset.location)
import glob
yaml_files = glob.glob('data/roboflow_raw/**/data.yaml', recursive=True)
print('data.yaml 位置：', yaml_files)
for split in ['train', 'valid', 'test']:
    path = f'data/roboflow_raw/{split}/images'
    if os.path.exists(path):
        print(f'  {split}: {len(os.listdir(path))} images')

In [ ]:
# ── Cell 5：類別對應 42 → 5 ────────────────────────────────────────────────────
import yaml, glob, os, shutil, pathlib

TARGET_NAMES = ['寶特瓶', '鐵鋁罐', '紙餐盒', '塑膠袋', '一般垃圾']

NAME_MAP = {
    'plastic bottle':   0, 'milk bottle': 0, 'plastic can': 0,
    'plastic canister': 0, 'plastic cup': 0,
    'aluminum can': 1, 'tin': 1, 'scrap metal': 1, 'aerosols': 1,
    'food can': 1, 'drink can': 1, 'iron utensils': 1,
    'paper': 2, 'cardboard': 2, 'paper cups': 2, 'paper cup': 2,
    'paper bag': 2, 'disposable tableware': 2, 'papier mache': 2,
    'paper shavings': 2, 'cellulose': 2,
    'plastic bag': 3, 'zip plastic bag': 3, 'stretch film': 3,
    'combined plastic': 3, 'plastic film': 3,
}

yaml_path = glob.glob('data/roboflow_raw/**/data.yaml', recursive=True)[0]
with open(yaml_path) as f:
    orig_yaml = yaml.safe_load(f)

orig_names = orig_yaml.get('names', [])
if isinstance(orig_names, dict):
    orig_names = [orig_names[i] for i in sorted(orig_names.keys())]

print(f'原始類別數：{len(orig_names)}\n')
id_map = {}
for idx, name in enumerate(orig_names):
    target = NAME_MAP.get(name.strip().lower(), 4)
    id_map[idx] = target
    marker = '✓ ' if target < 4 else '  '
    print(f'{marker}{idx:2d}: {name:40s} → {target} {TARGET_NAMES[target]}')

out_dir     = pathlib.Path('data/roboflow_5class')
dataset_dir = pathlib.Path(yaml_path).parent
counters    = {i: 0 for i in range(5)}

for split in ['train', 'valid', 'test']:
    img_src = dataset_dir / split / 'images'
    lbl_src = dataset_dir / split / 'labels'
    if not img_src.exists():
        continue
    out_split = 'val' if split == 'valid' else split
    out_img = out_dir / out_split / 'images'
    out_lbl = out_dir / out_split / 'labels'
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    for img_path in img_src.glob('*.[jJpP][pPnN][gG]'):
        shutil.copy2(img_path, out_img / img_path.name)
        lbl_path     = lbl_src / (img_path.stem + '.txt')
        out_lbl_path = out_lbl / (img_path.stem + '.txt')
        if lbl_path.exists():
            new_lines = []
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts: continue
                    new_cls = id_map.get(int(parts[0]), 4)
                    counters[new_cls] += 1
                    new_lines.append(f'{new_cls} ' + ' '.join(parts[1:]))
            with open(out_lbl_path, 'w') as f:
                f.write('\n'.join(new_lines) + ('\n' if new_lines else ''))
        else:
            out_lbl_path.touch()

new_yaml = f"""path: {out_dir.resolve()}
train: train/images
val:   val/images
test:  test/images

nc: 5
names:
  0: 寶特瓶
  1: 鐵鋁罐
  2: 紙餐盒
  3: 塑膠袋
  4: 一般垃圾
"""
(out_dir / 'data.yaml').write_text(new_yaml, encoding='utf-8')

print('\n✅ 類別對應完成')
print('各類別 bbox 數量：')
for cid, cnt in counters.items():
    print(f'  {cid} {TARGET_NAMES[cid]}: {cnt}')

In [ ]:
# ── Cell 6：訓練 YOLO26m → waste_sorter_v6 ────────────────────────────────────
from ultralytics import YOLO
import shutil, os

DATA_YAML = 'data/roboflow_5class/data.yaml'
assert os.path.exists(DATA_YAML), f'找不到 {DATA_YAML}，請先執行 Cell 5'

model = YOLO('yolo26m.pt')

results = model.train(
    data=DATA_YAML,
    epochs=80,        # Kaggle 30h/週，可跑更多
    imgsz=640,
    batch=16,
    device=0,
    project='runs/train',
    name='waste_sorter_v6',
    exist_ok=True,
    patience=20,      # 恢復 20，比 Colab 版更充分
    cos_lr=True,
    lr0=1e-3,
    lrf=1e-2,
    mosaic=1.0,
    copy_paste=0.3,   # 複製貼上增強 — 改善塑膠袋、寶特瓶少數類別
    fliplr=0.5,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    hsv_s=0.7,
    hsv_v=0.4,
    mixup=0.1,
    cls=0.3,          # 降低分類損失權重，減少對鐵鋁罐的偏向
)

# 輸出到 /kaggle/working/ 自動保存
src = 'runs/train/waste_sorter_v6/weights/best.pt'
dst = '/kaggle/working/best_v6.pt'
shutil.copy(src, dst)

mAP = results.results_dict.get('metrics/mAP50(B)', 'N/A')
print(f'\n訓練完成！mAP@50 = {mAP}')
print(f'✅ best_v6.pt 已存至 /kaggle/working/（右側 Output 欄可下載）')

In [ ]:
# ── Cell 7：匯出 ONNX（可選，在 Kaggle 直接匯出）─────────────────────────────
from ultralytics import YOLO
import shutil

model = YOLO('/kaggle/working/best_v6.pt')
model.export(format='onnx', imgsz=416, simplify=True, opset=12)

import glob
onnx_files = glob.glob('**/*.onnx', recursive=True)
print('ONNX 檔案：', onnx_files)

if onnx_files:
    shutil.copy(onnx_files[0], '/kaggle/working/best_v6.onnx')
    print('✅ best_v6.onnx 已存至 /kaggle/working/')
    print('\n接著在 Jetson 執行：')
    print('/usr/src/tensorrt/bin/trtexec --onnx=models/best_v6.onnx --saveEngine=models/best_v6.engine --fp16')